### step1: Setup

In [20]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
from agents import Agent, Runner, function_tool
import trafilatura


load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])


MODEL = "gpt-4.1-mini"

sk-proj-


### Step 2 : Define the tools

In [29]:
@function_tool
def search_web(query: str) ->str:
    """ Search the web using DubkDuckGo browser. Reeturn 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f" \u2705 search_web: Got results for {query}")
    return json.dumps(results, indent=2) # this step is to covert list into text as LLM underestands only text and not the list

@function_tool
def fetch_url(url:str) -> str:
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 fetch_rul: Got: {len(text)} chars from {url[60:]}")
            return text
    print(f" \u274C failed to fech or extract text from url: {url}")
    return f" Could not extract text from {url}. Try a different source"

In [31]:
from openai import OpenAI
openai_client = OpenAI()

@function_tool
def generate_image(prompt: str)-> str:
    """Generate an image using DALL-E 3. The prompt should be a detailed visual description"""
    print(f"  🎨 🖌️generate_image: {prompt[:60]}")
    response = openai_client.images.generate(
        model = "dall-e-3",
        prompt = prompt,
        size = "1792x1024",
        quality = "hd",
        style = "natural",
        n=1
    )
    image_url = response.data[0].url
    print(f"   ✅ genrate_image: Done")
    return f"Image generated successfully: {image_url}"

### Step 3: The Agents
### This tells the LLM who it is and how to behave. The key things:

### 1) What it job is?
### 2) What tools it has?
### 3) What process to follow?
### 4) what output format to product?

#### Research Agent

In [22]:

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

***IMPORTANT:
After each search with the search_web, you MUST first explain your reasoning:
- Which URLs look more relevant and why
- Which one you will fetch and why
- Which ones you are skipping nd why

Only AFTER writing out your resoning you sould call fetch_url
***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather informaton from at least 6 distinct sources before delviering your brief.
if you have a fewer than 6 sources then keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

research_agent = Agent(
    name = "Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_url]
)

#### Image Generator Agent

In [ ]:
IMAGE_AGENT_PROMPT = """
You are an impage prompt specialist. Given a topic and content summary, 
craft a detailed DALL-E prompt for a hero image.

Rules for your DALL-E prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logs, or words in the image
- No human faces or recoginizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt. One image only.
"""

image_agent = Agent(
    name = "Image Generator Agent",
    instructions=IMAGE_AGENT_PROMPT,
    model = MODEL,
    tools = [generate_image]
)

#### Orchestration Agent

In [23]:
# my version, vibe codiing
ORCHESTRATION_AGENT_PROMPT = """

You are a Technical Writer Orchestrator Agent.

## 1) Your Job
Your responsibility is to orchestrate the creation of technical content by coordinating between two specialized tools:
- Research Agent (for content generation)
- Image Generator Agent (for visuals)

You do NOT produce the final polished document yourself.
Instead, you:
- Break down the user’s request into structured sections
- Delegate content generation to the Research Agent
- Identify where visuals are needed and call the Image Generator Agent
- Collect and organize all outputs
- Hand off a complete, structured draft to the Writer Agent

Your goal is to ensure the Writer Agent receives high-quality, well-structured, and complete inputs.

---

## 2) Tools Available

### Tool 1: Research Agent
Use this tool to generate:
- Explanations
- Definitions
- Technical breakdowns
- Comparisons
- Examples

Input should include:
- Clear topic
- Audience level (e.g., beginner, intermediate, senior engineer)
- Specific instructions on depth and structure

---

### Tool 2: Image Generator Agent
Use this tool to generate:
- Architecture diagrams
- Workflow diagrams
- Conceptual visuals

Input should include:
- Clear description of the diagram
- Components to include
- Relationships/flow
- Style guidance (clean, labeled, technical)

---

## 3) Process to Follow

### Step 1: Understand the Request
- Identify the topic, audience, and expected depth
- Determine if the task is explanatory, architectural, or comparative

---

### Step 2: Create a Structured Outline
Break the topic into logical sections (e.g.):
- Overview
- Core Concepts
- Architecture / Workflow
- Use Cases
- Trade-offs / Best Practices

---

### Step 3: For Each Section

#### 3a. Call Research Agent
- Generate structured, high-quality technical content
- Ensure clarity and completeness

#### 3b. Decide if Visual is Needed
Call Image Generator Agent if:
- The section involves architecture
- The workflow is complex
- A diagram significantly improves understanding

Avoid unnecessary visuals.

---

### Step 4: Collect and Organize Outputs
- Store research outputs per section
- Store images with corresponding sections
- Ensure logical flow and no missing sections

---

### Step 5: Validate Completeness
Before handoff, ensure:
- All sections are covered
- Content depth is sufficient
- Diagrams exist where needed
- No redundancy or gaps

---

## 4) Output Format

Produce a structured handoff for the Writer Agent in the following JSON format:

{
  "title": "<document title>",
  "audience": "<target audience>",
  "sections": [
    {
      "section_title": "<section name>",
      "content": "<research agent output>",
      "visual": {
        "required": true/false,
        "description": "<what the diagram represents>",
        "image_output": "<image generator output or null>"
      }
    }
  ],
  "notes_for_writer": [
    "Any instructions for tone, flow, or emphasis",
    "Highlight areas needing simplification or storytelling",
    "Mention any assumptions or gaps if present"
  ]
}

---

## Behavioral Guidelines

- Always prefer structured delegation over self-generation
- Be precise and explicit in tool prompts
- Ensure outputs are consistent across sections
- Think like an editor assembling inputs, not an author writing prose
- Optimize for clarity, completeness, and usability by the Writer Agent
"""

## Optimize output
ORCHESTRATION_AGENT_PROMPT = """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.
"""

tool_research_agent = research_agent.as_tool(
    tool_name="research_agent",
    tool_description = "Research a topic and return a brief with key facts, statistics, theemes, and source URLs. Pass the topic as input.",
    max_turns=25
)

orchestrator_agent = Agent(
    name = "Orchestrator Agent",
    model="o4-mini", ## reasoning
    instructions=ORCHESTRATION_AGENT_PROMPT,
    tools=[tool_research_agent]
    
)

### Lets Run it

In [28]:
from agents import trace

# with trace("Article Writer"):
with trace("Article Writer", group_id="Learning AI Engineering",
           metadata={"experimentId": "001"}):
    result = await Runner.run(
        orchestrator_agent,
        input = "Research a following topic and provide a comprehensive result: how AI will use in health industries in 2030",
        max_turns=30
    )

 ✅ search_web: Got results for future uses of AI in health industry by 2030
 ✅ fetch_rul: Got: 99483 chars from lthcare
 ✅ fetch_rul: Got: 24395 chars from ligence-future
 ✅ fetch_rul: Got: 25023 chars from re/
 ✅ search_web: Got results for how AI will be used in healthcare by 2030 expert reports
 ✅ search_web: Got results for AI in healthcare 2030 report site:.gov OR site:.org OR site:.edu
 ❌ failed to fech or extract text from url: https://www.healthit.gov/wp-content/uploads/2025/09/Data-Brief-80-Hospital-Trends-in-the-Use-Evaluation-and-Governance-of-Predictive-AI-2023-2024_508.pdf
 ❌ failed to fech or extract text from url: https://www.hhs.gov/programs/topic-sites/ai/strategy-implementation/index.html
 ❌ failed to fech or extract text from url: https://digitalgovernmenthub.org/wp-content/uploads/2025/02/2025-HHS-AI-Strategic-Plan_Full_508.pdf
 ✅ search_web: Got results for AI in healthcare future 2030 site:who.int OR site:oecd.org OR site:nature.com
 ❌ failed to fech or extract te

Error getting response: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-tJxVAWtFl1xOvkLoKMxxib7x on tokens per min (TPM): Limit 200000, Used 174195, Requested 39078. Please try again in 3.981s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}. (request_id: req_7996cff9cb5246ba82e0a532d52892fb)


 ✅ search_web: Got results for use of AI in health industries 2030
 ❌ failed to fech or extract text from url: https://www.startus-insights.com/innovators-guide/ai-in-healthcare/
 ✅ fetch_rul: Got: 1329 chars from 
 ✅ search_web: Got results for AI in healthcare market forecast 2030
 ✅ fetch_rul: Got: 24992 chars from arket
 ❌ failed to fech or extract text from url: https://www.weforum.org/stories/2025/08/ai-transforming-global-health/
 ✅ search_web: Got results for AI healthcare future predictions 2030
 ❌ failed to fech or extract text from url: https://www.weforum.org/stories/2020/01/future-of-artificial-intelligence-healthcare-delivery/
 ✅ search_web: Got results for AI healthcare 2030 trends report
 ✅ fetch_rul: Got: 5379 chars from -the-industry-forward
 ✅ fetch_rul: Got: 8249 chars from market/
 ✅ search_web: Got results for Applications of AI in healthcare industries by 2030
 ✅ fetch_rul: Got: 15858 chars from tions-of-ai-in-healthcare
 ❌ failed to fech or extract text from url

In [27]:
print(f"Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

Agent: Orchestrator Agent
-----


I’ve reviewed both research briefs and selected the second one as the more comprehensive overview. Below is the chosen briefing:

Research Brief: Applications of AI in Healthcare by 2030

1. Market Growth & Projections  
 • Global AI-in-Healthcare market size  
   – USD 21.66 B in 2025 → USD 110.61 B by 2030 (CAGR 38.6%)  
 • Agentic AI in healthcare segment  
   – USD 538.51 M in 2024 → USD 4.96 B by 2030 (CAGR 45.6%)  
 • Regional leadership  
   – North America: ~42–55% market share  
   – Fastest growth: Asia Pacific

2. Key Application Areas  
 • Diagnostics & Early Detection  
   – AI-powered medical imaging, cardiac AI  
 • Personalized Medicine  
   – Genomic analysis, tailored treatment planning  
 • Clinical Decision Support  
   – Predictive analytics, triage automation  
 • Drug Discovery & Development  
   – Molecule screening, trial optimization  
 • Remote Patient Monitoring & Virtual Care  
   – Wearables, tele-ICU, chronic disease management  
 • Administrative Workflows  
   – Revenue cycle management, claims processing  

3. Technology Trends  
 • Machine Learning & Deep Learning models  
 • Cloud-based AI deployment  
 • Multi-agent and generative AI systems  
 • Large language models for clinical documentation  

4. Driving Forces  
 • Rising healthcare data volumes  
 • Demand for improved diagnostic accuracy  
 • Need for cost reduction and operational efficiency  
 • Patient demand for personalized care  

5. Challenges & Barriers  
 • Practitioner reluctance and workflow integration  
 • Data privacy, security, and interoperability  
 • Ethical concerns and bias mitigation  
 • Regulatory frameworks and liability issues  
 • Shortage of skilled AI professionals  

6. Notable Statistics  
 • AI imaging market share ~19% of total AI healthcare revenue  
 • CAGR in integrated AI-health solutions ~39–46%  
 • Significant uptick in generative AI pilots among U.S. health systems in 2024–25  

7. Leading Industry Players  
 • Philips, Siemens Healthineers, GE Healthcare  
 • Microsoft Azure, AWS, IBM Watson Health  
 • NVIDIA, Oracle, Thoughtful Automation, Hippocratic AI  

8. Sources  
 • MarketsandMarkets:  
   https://www.marketsandmarkets.com/Market-Reports/artificial-intelligence-healthcare-market-54679303.html  
 • Statista on AI in Healthcare:  
   https://www.statista.com/topics/10011/ai-in-healthcare/  
 • JAPMI “AI and Healthcare in 2030”:  
   https://japmi.org/index.php/japmi/article/view/23  
 • Grand View Research Agentic AI Report:  
   https://www.grandviewresearch.com/industry-analysis/agentic-ai-healthcare-market-report